# Task 3 — DeBERTa-v3-base Fine-tuning (AI Text Detection)

> **Mục tiêu:** Fine-tune `microsoft/deberta-v3-base` với các kỹ thuật tối ưu:
> - **LLRD** (Layerwise Learning Rate Decay) — tránh catastrophic forgetting
> - **AMP** (Automatic Mixed Precision) — tăng tốc trên GPU
> - **Gradient Accumulation** — mô phỏng batch size lớn hơn
> - **Cosine Scheduler with Warmup** — learning rate ổn định
> - **Early Stopping + Checkpointing** — lưu model tốt nhất

In [4]:
# ── Standard library
import os, sys, gc, time, warnings, math, pathlib
from pathlib import Path
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')

# ── Scientific stack
import numpy  as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ── PyTorch
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader

# ── Hugging Face
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_cosine_schedule_with_warmup,
)

# ── tqdm
from tqdm.auto import tqdm

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA ok : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')


Python  : 3.12.12
PyTorch : 2.10.0+cu128
CUDA ok : True
GPU     : Tesla T4
VRAM    : 14.6 GB


## 1. Cấu hình (CFG)

In [6]:
class CFG:
    # Đường dẫn trên Kaggle
    base_dir   = Path('/kaggle/working') # Nơi lưu checkpoint, logs
    input_dir  = Path('/kaggle/input/datasets/thedrcat/daigt-v2-train-dataset') # Dataset của bạn
    deberta_dir = Path('/kaggle/input/notebooks/rushslow/microsoft-deberta-v3-base') # Model DeBERTa offline


    # ── Data
    text_col   = 'text_clean'
    label_col  = 'label'
    val_fold   = 0          # Fold dùng để validation
    n_folds    = 5
    seed       = 42

    # ── Model
    model_name = 'microsoft/deberta-v3-base'
    max_length = 512
    num_labels = 2
    pooling    = 'mean'     # 'mean' hoặc 'cls'
    dropout    = 0.1

    # ── Training
    epochs          = 5
    batch_size      = 16
    grad_accum      = 2     # effective batch = 8 * 4 = 32
    max_grad_norm   = 1.0
    num_workers     = 2     # Windows: luôn để 0

    # ── Optimizer / Scheduler
    lr_head        = 1e-4   # Learning rate cho classifier head
    lr_backbone    = 2e-5   # Learning rate cho backbone DeBERTa
    llrd_factor    = 0.9    # Mỗi layer giảm: lr * 0.9^(depth)
    weight_decay   = 0.01
    warmup_ratio   = 0.1    # 10% steps đầu warmup
    eps            = 1e-6
    betas          = (0.9, 0.999)

    # ── AMP
    use_amp        = True   # Mixed precision (chỉ dùng khi có CUDA)

    # ── Early Stopping
    patience       = 2      # Số epoch liên tiếp không cải thiện
    min_delta      = 1e-4   # Ngưỡng cải thiện tối thiểu

# Seed everything
def seed_everything(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(CFG.seed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = CFG.use_amp and torch.cuda.is_available()

print(f'Device  : {DEVICE}')
print(f'Use AMP : {USE_AMP}')
print(f'Eff. batch size : {CFG.batch_size * CFG.grad_accum}')


Device  : cuda
Use AMP : True
Eff. batch size : 32


## 2. Load Dữ liệu
> Tải dữ liệu đã được xử lý từ Task 2 (`train_clean.pkl`). File này phải tồn tại trước khi chạy Task 3.

In [9]:
pkl_path = Path('/kaggle/input/datasets/caosfourn/train-clean/train_clean.pkl')
csv_path = Path('/kaggle/input/datasets/thedrcat/daigt-v2-train-dataset/train_v2_drcat_02.csv')

if pkl_path.exists():
    df = pd.read_pickle(pkl_path)
    print(f'Loaded từ pickle: {pkl_path.name} — shape={df.shape}')
else:
    raise FileNotFoundError(
        f'Không tìm thấy {pkl_path}.\n'
        f'Hãy chạy Task 2 (task2_pipeline.ipynb) trước để tạo file này!'
    )

# Đảm bảo text_clean tồn tại
if CFG.text_col not in df.columns:
    print('Cột text_clean chưa có, dùng cột text gốc...')
    df[CFG.text_col] = df['text']

# Đảm bảo fold đã được tạo
if 'fold' not in df.columns:
    from sklearn.model_selection import StratifiedKFold
    df['fold'] = -1
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    for fold_id, (_, val_idx) in enumerate(skf.split(df, df[CFG.label_col])):
        df.loc[val_idx, 'fold'] = fold_id
    print(f'Đã tạo {CFG.n_folds} folds')

train_df = df[df['fold'] != CFG.val_fold].reset_index(drop=True)
valid_df = df[df['fold'] == CFG.val_fold].reset_index(drop=True)

print(f'Train : {len(train_df):,} mẫu (Human={( train_df[CFG.label_col]==0).sum():,} | AI={(train_df[CFG.label_col]==1).sum():,})')
print(f'Valid : {len(valid_df):,} mẫu (Human={(valid_df[CFG.label_col]==0).sum():,} | AI={(valid_df[CFG.label_col]==1).sum():,})')


Loaded từ pickle: train_clean.pkl — shape=(44868, 3)
Train : 35,894 mẫu (Human=21,896 | AI=13,998)
Valid : 8,974 mẫu (Human=5,475 | AI=3,499)


## 3. Tokenizer & Dataset

In [10]:
print(f'Đang tải tokenizer: {CFG.model_name} ...')
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)
print(f'Tokenizer sẵn sàng! Vocab size: {tokenizer.vocab_size:,}')


Đang tải tokenizer: microsoft/deberta-v3-base ...
Tokenizer sẵn sàng! Vocab size: 128,000


In [11]:
class AITextDataset(Dataset):
    """Custom PyTorch Dataset cho bài toán phân loại Human/AI."""

    def __init__(
        self,
        texts     : List[str],
        tokenizer ,
        max_length: int,
        labels    : Optional[List[int]] = None,
    ):
        assert len(texts) > 0, 'Dataset rỗng!'
        if labels is not None:
            assert len(texts) == len(labels)
        self.texts      = texts
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.labels     = labels
        self.is_test    = (labels is None)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        encoding = self.tokenizer(
            self.texts[idx],
            max_length            = self.max_length,
            padding               = 'max_length',
            truncation            = True,
            return_tensors        = 'pt',
            return_attention_mask = True,
        )
        item: Dict[str, torch.Tensor] = {
            'input_ids'     : encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        if not self.is_test:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = AITextDataset(
    texts      = train_df[CFG.text_col].tolist(),
    tokenizer  = tokenizer,
    max_length = CFG.max_length,
    labels     = train_df[CFG.label_col].tolist(),
)
valid_dataset = AITextDataset(
    texts      = valid_df[CFG.text_col].tolist(),
    tokenizer  = tokenizer,
    max_length = CFG.max_length,
    labels     = valid_df[CFG.label_col].tolist(),
)

train_loader = DataLoader(
    train_dataset, batch_size=CFG.batch_size,
    shuffle=True,  num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(), drop_last=False,
)
valid_loader = DataLoader(
    valid_dataset, batch_size=CFG.batch_size * 2,
    shuffle=False, num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(), drop_last=False,
)

print(f'Train dataset : {len(train_dataset):,} samples | {len(train_loader):,} batches')
print(f'Valid dataset : {len(valid_dataset):,} samples | {len(valid_loader):,} batches')


Train dataset : 35,894 samples | 2,244 batches
Valid dataset : 8,974 samples | 281 batches


## 4. Kiến trúc mô hình: DeBERTaDetector

```
Input → DeBERTa backbone → Mean Pooling → Dropout → Linear(hidden, 2) → Logits
```

**Mean Pooling** tốt hơn `[CLS]` vì:
- Tổng hợp thông tin toàn bộ câu, không chỉ token đầu tiên
- Ổn định hơn khi văn bản dài (essay, essay-like text)

In [14]:
class MeanPooling(nn.Module):
    """Mean pooling có tính đến attention mask (bỏ padding)."""

    def forward(
        self,
        hidden_state  : torch.Tensor,  # [B, L, H]
        attention_mask: torch.Tensor,  # [B, L]
    ) -> torch.Tensor:                 # [B, H]
        mask_expanded = attention_mask.unsqueeze(-1).float()  # [B, L, 1]
        sum_hidden    = (hidden_state * mask_expanded).sum(dim=1)  # [B, H]
        sum_mask      = mask_expanded.sum(dim=1).clamp(min=1e-9)   # [B, 1]
        return sum_hidden / sum_mask


class DeBERTaDetector(nn.Module):
    """
    Mô hình phát hiện AI-generated text dựa trên DeBERTa-v3-base.

    Kiến trúc:
        DeBERTa backbone → MeanPooling → Dropout → Linear → logits
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg    = cfg
        self.config = AutoConfig.from_pretrained(
            cfg.model_name,
            num_labels           = cfg.num_labels,
            hidden_dropout_prob  = cfg.dropout,
            attention_probs_dropout_prob = cfg.dropout,
        )
        self.backbone  = AutoModel.from_pretrained(
            cfg.model_name, config=self.config
        )
        self.pooler    = MeanPooling()
        hidden_size    = self.config.hidden_size
        self.dropout   = nn.Dropout(p=cfg.dropout)
        self.classifier = nn.Linear(hidden_size, cfg.num_labels)
        self._init_weights(self.classifier)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(
        self,
        input_ids      : torch.Tensor,
        attention_mask : torch.Tensor,
        token_type_ids : Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        outputs = self.backbone(
            input_ids       = input_ids,
            attention_mask  = attention_mask,
            token_type_ids  = token_type_ids,
        )
        pooled  = self.pooler(outputs.last_hidden_state, attention_mask)
        pooled  = self.dropout(pooled)
        logits  = self.classifier(pooled)
        return logits   # [B, 2]


# Khởi tạo model
model = DeBERTaDetector(CFG).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {CFG.model_name}')
print(f'Tổng parameters: {n_params / 1e6:.1f}M')
print(f'Hidden size     : {model.config.hidden_size}')
print(f'Num layers      : {model.config.num_hidden_layers}')


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

Model: microsoft/deberta-v3-base
Tổng parameters: 183.8M
Hidden size     : 768
Num layers      : 12


## 5. Optimizer với LLRD (Layerwise Learning Rate Decay)

**LLRD** gán learning rate khác nhau cho mỗi layer:
- **Classifier head:** `lr_head = 1e-4` (học nhanh nhất)
- **Embedding layer:** `lr_backbone * factor^(n_layers)` (học chậm nhất)
- Mỗi layer Transformer: `lr = lr_backbone * factor^(n_layers - layer_idx)`

→ Các layer gần input giữ nguyên pre-trained weights, tránh catastrophic forgetting.

In [15]:
def get_optimizer_groups(model: nn.Module, cfg) -> List[Dict]:
    """
    Tạo parameter groups với LLRD cho AdamW.
    Returns: list of {'params': ..., 'lr': ..., 'weight_decay': ...}
    """
    # Không áp dụng weight decay cho bias và LayerNorm
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']

    # ── Head parameters (lr cao nhất)
    groups = [
        {
            'params'      : [p for n, p in model.classifier.named_parameters()
                             if p.requires_grad and not any(nd in n for nd in no_decay)],
            'lr'          : cfg.lr_head,
            'weight_decay': cfg.weight_decay,
        },
        {
            'params'      : [p for n, p in model.classifier.named_parameters()
                             if p.requires_grad and any(nd in n for nd in no_decay)],
            'lr'          : cfg.lr_head,
            'weight_decay': 0.0,
        },
    ]

    # ── Backbone LLRD: Transformer layers
    n_layers = model.config.num_hidden_layers
    for layer_idx in range(n_layers - 1, -1, -1):
        depth  = n_layers - 1 - layer_idx
        lr_cur = cfg.lr_backbone * (cfg.llrd_factor ** depth)
        layer  = model.backbone.encoder.layer[layer_idx]
        named  = list(layer.named_parameters())
        groups += [
            {
                'params'      : [p for n, p in named
                                 if p.requires_grad and not any(nd in n for nd in no_decay)],
                'lr'          : lr_cur,
                'weight_decay': cfg.weight_decay,
            },
            {
                'params'      : [p for n, p in named
                                 if p.requires_grad and any(nd in n for nd in no_decay)],
                'lr'          : lr_cur,
                'weight_decay': 0.0,
            },
        ]

    # ── Embedding parameters (lr thấp nhất)
    lr_emb = cfg.lr_backbone * (cfg.llrd_factor ** n_layers)
    emb_named = list(model.backbone.embeddings.named_parameters())
    groups += [
        {
            'params'      : [p for n, p in emb_named
                             if p.requires_grad and not any(nd in n for nd in no_decay)],
            'lr'          : lr_emb,
            'weight_decay': cfg.weight_decay,
        },
        {
            'params'      : [p for n, p in emb_named
                             if p.requires_grad and any(nd in n for nd in no_decay)],
            'lr'          : lr_emb,
            'weight_decay': 0.0,
        },
    ]

    # Lọc groups rỗng
    return [g for g in groups if len(g['params']) > 0]


optimizer_groups = get_optimizer_groups(model, CFG)
optimizer = torch.optim.AdamW(
    optimizer_groups,
    betas   = CFG.betas,
    eps     = CFG.eps,
)

lrs = [g['lr'] for g in optimizer_groups]
print(f'Optimizer: AdamW | Số groups: {len(optimizer_groups)}')
print(f'LR range : {min(lrs):.2e} → {max(lrs):.2e}')


Optimizer: AdamW | Số groups: 28
LR range : 5.65e-06 → 1.00e-04


## 6. Learning Rate Scheduler

In [16]:
total_steps   = (len(train_loader) // CFG.grad_accum) * CFG.epochs
warmup_steps  = int(total_steps * CFG.warmup_ratio)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)

scaler = GradScaler(enabled=USE_AMP)

print(f'Total optimizer steps : {total_steps:,}')
print(f'Warmup steps          : {warmup_steps:,} ({CFG.warmup_ratio*100:.0f}%)')
print(f'Epochs                : {CFG.epochs}')
print(f'GradScaler enabled    : {USE_AMP}')


Total optimizer steps : 5,610
Warmup steps          : 561 (10%)
Epochs                : 5
GradScaler enabled    : True


## 7. Hàm train / evaluate

In [17]:
criterion = nn.CrossEntropyLoss()


def train_one_epoch(
    model, loader, optimizer, scheduler, scaler, device, epoch, cfg
) -> Dict[str, float]:
    model.train()
    total_loss, n_correct, n_total = 0.0, 0, 0
    optimizer.zero_grad()
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} [Train]', leave=False, ncols=100)

    for step, batch in enumerate(pbar):
        input_ids      = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        with autocast(enabled=USE_AMP):
            logits = model(input_ids, attention_mask, token_type_ids)
            loss   = criterion(logits, labels) / cfg.grad_accum

        scaler.scale(loss).backward()

        # Gradient accumulation step
        if (step + 1) % cfg.grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        # Tracking
        total_loss += loss.item() * cfg.grad_accum
        preds       = logits.argmax(dim=-1)
        n_correct  += (preds == labels).sum().item()
        n_total    += len(labels)

        pbar.set_postfix({'loss': f'{total_loss/(step+1):.4f}',
                          'acc' : f'{n_correct/n_total:.4f}',
                          'lr'  : f'{scheduler.get_last_lr()[0]:.2e}'})

    return {
        'loss': total_loss / len(loader),
        'acc' : n_correct  / n_total,
    }


@torch.no_grad()
def evaluate(
    model, loader, device
) -> Dict[str, float]:
    model.eval()
    total_loss, n_correct, n_total = 0.0, 0, 0
    all_probs, all_labels = [], []
    pbar = tqdm(loader, desc='        [Valid]', leave=False, ncols=100)

    for batch in pbar:
        input_ids      = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        token_type_ids = batch.get('token_type_ids')
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)

        with autocast(enabled=USE_AMP):
            logits = model(input_ids, attention_mask, token_type_ids)
            loss   = criterion(logits, labels)

        probs  = torch.softmax(logits, dim=-1)[:, 1]  # P(AI)
        preds  = logits.argmax(dim=-1)

        total_loss += loss.item()
        n_correct  += (preds == labels).sum().item()
        n_total    += len(labels)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_probs)
    return {
        'loss': total_loss / len(loader),
        'acc' : n_correct  / n_total,
        'auc' : auc,
        'probs' : np.array(all_probs),
        'labels': np.array(all_labels),
    }


## 8. Early Stopping & Checkpointing

In [18]:
class EarlyStopping:
    """Dừng training sớm nếu AUC không cải thiện sau `patience` epochs."""

    def __init__(self, patience: int = 3, min_delta: float = 1e-4, mode: str = 'max'):
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self.best_score = None
        self.counter    = 0
        self.stop       = False

    def step(self, score: float) -> bool:
        """Returns True nếu nên dừng."""
        improved = (
            self.best_score is None or
            (self.mode == 'max' and score > self.best_score + self.min_delta) or
            (self.mode == 'min' and score < self.best_score - self.min_delta)
        )
        if improved:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop

early_stopping = EarlyStopping(patience=CFG.patience, min_delta=CFG.min_delta, mode='max')
print('EarlyStopping khởi tạo thành công!')
print(f'  Patience  : {CFG.patience} epochs')
print(f'  Min delta : {CFG.min_delta}')


EarlyStopping khởi tạo thành công!
  Patience  : 2 epochs
  Min delta : 0.0001


## 9. Training Loop chính

In [20]:
CKPT_PATH = CFG.base_dir / f'deberta_fold{CFG.val_fold}_best.pt'
LOG_PATH  = CFG.base_dir / f'train_log_fold{CFG.val_fold}.csv'


history = []
best_auc = 0.0

print('=' * 65)
print(f'  BẮT ĐẦU TRAINING — {CFG.model_name}')
print(f'  Fold {CFG.val_fold} | Epochs {CFG.epochs} | Device: {DEVICE}')
print('=' * 65)

t0 = time.time()

for epoch in range(CFG.epochs):
    t_epoch = time.time()

    # ── Train
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler,
        DEVICE, epoch, CFG
    )

    # ── Evaluate
    valid_metrics = evaluate(model, valid_loader, DEVICE)

    elapsed = time.time() - t_epoch
    current_auc = valid_metrics['auc']

    # ── Log
    row = {
        'epoch'     : epoch + 1,
        'train_loss': train_metrics['loss'],
        'train_acc' : train_metrics['acc'],
        'valid_loss': valid_metrics['loss'],
        'valid_acc' : valid_metrics['acc'],
        'valid_auc' : current_auc,
        'time_s'    : elapsed,
    }
    history.append(row)

    print(f"\nEpoch [{epoch+1}/{CFG.epochs}] "
          f"({elapsed/60:.1f} min)")
    print(f"  Train | Loss: {train_metrics['loss']:.4f}  Acc: {train_metrics['acc']:.4f}")
    print(f"  Valid | Loss: {valid_metrics['loss']:.4f}  Acc: {valid_metrics['acc']:.4f}  AUC: {current_auc:.4f}", end='')

    # ── Checkpoint nếu tốt hơn
    if current_auc > best_auc:
        best_auc = current_auc
        torch.save({
            'epoch'      : epoch + 1,
            'state_dict' : model.state_dict(),
            'best_auc'   : best_auc,
            'cfg'        : CFG.__dict__ if hasattr(CFG, '__dict__') else {},
        }, CKPT_PATH)
        print(' ← BEST (saved)')
    else:
        print()

    # ── Early stopping check
    if early_stopping.step(current_auc):
        print(f'\nEarly stopping triggered sau epoch {epoch+1} (best AUC={best_auc:.4f})')
        break

    # ── Memory cleanup
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ── Save log
history_df = pd.DataFrame(history)
history_df.to_csv(LOG_PATH, index=False)

total_time = (time.time() - t0) / 60
print('\n' + '=' * 65)
print(f'  TRAINING HOÀN TẤT!')
print(f'  Best Valid AUC : {best_auc:.4f}')
print(f'  Checkpoint     : {CKPT_PATH}')
print(f'  Tổng thời gian : {total_time:.1f} phút')
print('=' * 65)


  BẮT ĐẦU TRAINING — microsoft/deberta-v3-base
  Fold 0 | Epochs 5 | Device: cuda


Epoch 1 [Train]:   0%|                                                     | 0/2244 [00:00<?, ?it/s]

ValueError: Attempting to unscale FP16 gradients.

## 10. Visualize Training History

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120

if len(history) > 0:
    hist_df = pd.DataFrame(history)
    epochs_ran = hist_df['epoch'].tolist()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('DeBERTa Training History', fontsize=14, fontweight='bold')

    # Loss
    axes[0].plot(epochs_ran, hist_df['train_loss'], 'b-o', label='Train Loss', lw=2)
    axes[0].plot(epochs_ran, hist_df['valid_loss'], 'r-s', label='Valid Loss', lw=2)
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(epochs_ran, hist_df['train_acc'], 'b-o', label='Train Acc', lw=2)
    axes[1].plot(epochs_ran, hist_df['valid_acc'], 'r-s', label='Valid Acc', lw=2)
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    axes[1].grid(alpha=0.3)

    # AUC
    axes[2].plot(epochs_ran, hist_df['valid_auc'], 'g-D', label='Valid AUC', lw=2, markersize=8)
    best_ep = hist_df.loc[hist_df['valid_auc'].idxmax(), 'epoch']
    axes[2].axvline(best_ep, color='red', ls='--', label=f'Best epoch={best_ep}')
    axes[2].set_title('ROC-AUC'); axes[2].set_xlabel('Epoch'); axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    fig_path = CFG.output_dir / f'training_history_fold{CFG.val_fold}.png'
    plt.savefig(fig_path, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')
    print()
    print(hist_df[['epoch','train_loss','valid_loss','train_acc','valid_acc','valid_auc']].to_string(index=False))
else:
    print('Chưa có history. Hãy chạy training loop trước.')


## 11. Load Best Checkpoint & Sinh OOF Predictions

In [ ]:
# Load checkpoint tốt nhất
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded checkpoint: epoch={ckpt["epoch"]} | best_auc={ckpt["best_auc"]:.4f}')

# Sinh OOF predictions (cần cho ensemble ở Task 4)
oof_metrics  = evaluate(model, valid_loader, DEVICE)

oof_df = valid_df[['text_clean', CFG.label_col, 'fold']].copy()
oof_df['oof_prob_ai'] = oof_metrics['probs']
oof_df['oof_pred']    = (oof_df['oof_prob_ai'] >= 0.5).astype(int)

oof_path = CFG.output_dir / f'oof_deberta_fold{CFG.val_fold}.pkl'
oof_df.to_pickle(oof_path)

print(f'\n[OOF] Fold {CFG.val_fold}:')
print(f'  AUC   : {oof_metrics["auc"]:.4f}')
print(f'  ACC   : {oof_metrics["acc"]:.4f}')
print(f'  Saved : {oof_path}')
